# Kafka 연결 테스트

Kafka 브로커 연결 및 Topic 확인

In [ ]:
import sys
sys.path.insert(0, '../..')

from kafka import KafkaAdminClient, KafkaConsumer
from kafka.admin import ConfigResource, ConfigResourceType
from config.settings import KAFKA_BROKERS

print("📌 Kafka 브로커 정보:")
print(f"  Brokers: {KAFKA_BROKERS}")

## 1. 브로커 연결 테스트

In [ ]:
from kafka import KafkaAdminClient

try:
    admin_client = KafkaAdminClient(bootstrap_servers=','.join(KAFKA_BROKERS))
    admin_client.close()
    print("✅ Kafka 브로커 연결 성공!")
except Exception as e:
    print(f"❌ 연결 실패: {e}")

## 2. Topic 목록 확인

In [ ]:
from kafka.admin import KafkaAdminClient

try:
    admin_client = KafkaAdminClient(bootstrap_servers=','.join(KAFKA_BROKERS))
    metadata = admin_client.describe_cluster()
    
    print(f"📊 클러스터 정보:")
    print(f"  Brokers: {len(metadata['brokers'])}개")
    for broker_id in metadata['brokers']:
        print(f"    - Broker {broker_id}")
    
    admin_client.close()
except Exception as e:
    print(f"❌ 조회 실패: {e}")

## 3. 주요 Topic 확인

In [ ]:
from kafka import KafkaConsumer

try:
    consumer = KafkaConsumer(bootstrap_servers=','.join(KAFKA_BROKERS))
    topics = consumer.topics()
    
    print(f"📋 Total Topics: {len(topics)}")
    print()
    
    important_topics = ['clickstream', 'error_data', 'perf-test']
    for topic in important_topics:
        if topic in topics:
            partitions = consumer.partitions_for_topic(topic)
            print(f"✅ {topic:<20} : {len(partitions)}개 partition")
        else:
            print(f"❌ {topic:<20} : (없음)")
    
    consumer.close()
except Exception as e:
    print(f"❌ 조회 실패: {e}")

## 4. error_data Topic 메시지 개수 확인

In [ ]:
from kafka import KafkaConsumer, TopicPartition

try:
    consumer = KafkaConsumer(
        bootstrap_servers=','.join(KAFKA_BROKERS),
        enable_auto_commit=False
    )
    
    topic = 'error_data'
    partitions = consumer.partitions_for_topic(topic)
    
    total_lag = 0
    print(f"📊 {topic} Topic 상태:")
    print()
    
    for partition in sorted(partitions):
        tp = TopicPartition(topic, partition)
        consumer.assign([tp])
        
        # End offset (마지막 메시지 위치)
        consumer.seek_to_end(tp)
        end_offset = consumer.position(tp)
        
        # Beginning offset
        consumer.seek_to_beginning(tp)
        beginning_offset = consumer.position(tp)
        
        messages = end_offset - beginning_offset
        total_lag += messages
        
        print(f"  Partition {partition}: {messages:>6,}개 메시지")
    
    print()
    print(f"✅ 총 메시지: {total_lag:,}개")
    
    consumer.close()
except Exception as e:
    print(f"❌ 조회 실패: {e}")